# Linear static analysis

The function [solve](../static.rst) solves a linear system of equations,
removing the null rows and columns of the stiffness matrix, which typically
correspond to constrained degrees of freedom. The values of the solution at
these degrees of freedom are zero. The code below is that of the test
`test_solve_with_null_columns` of `tests/test_static.py`:

In [1]:
import numpy as np
from scipy.sparse import csc_matrix

from structsolve import Analysis, solve

In [2]:
def test_solve_with_null_columns():
    """Solve a system where some DOFs are constrained (null rows/cols)"""
    n = 6
    K_dense = np.zeros((n, n))
    # Only DOFs 1,2,3,4 are active
    active = np.array([1, 2, 3, 4])
    for i in active:
        K_dense[i, i] = float(i + 1)
    K = csc_matrix(K_dense)
    f = np.zeros(n)
    f[active] = 1.0
    u = solve(K, f, silent=True)
    # DOFs 0 and 5 should remain zero
    assert u[0] == 0.0
    assert u[5] == 0.0
    for i in active:
        np.testing.assert_allclose(u[i], 1.0 / (i + 1))

In [3]:
test_solve_with_null_columns()
print('test_solve_with_null_columns: passed')

test_solve_with_null_columns: passed


Linear static analyses can also be run with [static](../static.rst), or
with the class [Analysis](../analysis.rst), which receives callables that
calculate the external force vector and the stiffness matrix, here those of
`tests/test_analysis.py`:

In [4]:
def _make_linear_system(n=10):
    """Create simple linear system callables"""
    K = csc_matrix(np.eye(n) * 3.0)
    f = np.ones(n) * 6.0

    def calc_fext(inc=1., silent=True):
        return f

    def calc_kC(c=None, NLgeom=False, silent=True):
        return K

    return calc_fext, calc_kC, K, f

In [5]:
def test_analysis_linear_static():
    """Test Analysis class for linear static analysis"""
    n = 10
    calc_fext, calc_kC, K, f = _make_linear_system(n)

    an = Analysis(calc_fext=calc_fext, calc_kC=calc_kC)
    increments, cs = an.static(NLgeom=False, silent=True)

    assert len(increments) == 1
    assert increments[0] == 1.0
    assert len(cs) == 1
    np.testing.assert_allclose(cs[0], 2.0 * np.ones(n))
    assert an.last_analysis == 'static'

In [6]:
test_analysis_linear_static()
print('test_analysis_linear_static: passed')

test_analysis_linear_static: passed
